In [1]:

from pathlib import Path

ROOT = Path("/kaggle/input/datasets")

DATASETS = {
    "CheXpert": ROOT / "ashery" / "chexpert",
    "NIH": Path("/kaggle/input"),
}

for name, path in DATASETS.items():
    print(f"\n{'='*60}")
    print(name)
    print("Path:", path)

    if not path.exists():
        print("Path not found. Checking matching folders...")

        if name == "NIH":
            matches = list(ROOT.glob("*/*NIH*"))
            for match in matches[:5]:
                print(match)
        continue

    for item in list(path.iterdir())[:30]:
        print(
            "DIR  " if item.is_dir() else "FILE ",
            item.name
        )

print("\nFast dataset structure inspection completed.")



CheXpert
Path: /kaggle/input/datasets/ashery/chexpert
DIR   train
FILE  train.csv
DIR   valid
FILE  valid.csv

NIH
Path: /kaggle/input
DIR   datasets

Fast dataset structure inspection completed.


In [2]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/input/datasets")

print("=== AVAILABLE DATASET FOLDERS ===")

for owner in ROOT.iterdir():
    if owner.is_dir():
        for dataset in owner.iterdir():
            if dataset.is_dir():
                print(f"{owner.name}/{dataset.name}")

# CheXpert metadata
chexpert_csv = Path(
    "/kaggle/input/datasets/ashery/chexpert/train.csv"
)

print("\n=== CHEXPERT TRAIN METADATA ===")

chexpert_df = pd.read_csv(chexpert_csv, nrows=5)

print("Columns:", chexpert_df.columns.tolist())
print("\nSample:")
print(chexpert_df.to_string(index=False))

# Find NIH metadata CSVs using folder names only
print("\n=== NIH METADATA FILES ===")

for owner in ROOT.iterdir():
    if not owner.is_dir():
        continue

    for dataset in owner.iterdir():
        if not dataset.is_dir():
            continue

        if "nih" in dataset.name.lower():
            print("\nDataset:", dataset.name)

            for csv_file in dataset.glob("*.csv"):
                print("CSV:", csv_file)
                sample = pd.read_csv(csv_file, nrows=3)
                print("Columns:", sample.columns.tolist())
                print(sample.head(2).to_string(index=False))

print("\nMetadata inspection completed.")


=== AVAILABLE DATASET FOLDERS ===
ashery/chexpert
nih-chest-xrays/data

=== CHEXPERT TRAIN METADATA ===
Columns: ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']

Sample:
                                                           Path    Sex  Age Frontal/Lateral AP/PA  No Finding  Enlarged Cardiomediastinum  Cardiomegaly  Lung Opacity  Lung Lesion  Edema  Consolidation  Pneumonia  Atelectasis  Pneumothorax  Pleural Effusion  Pleural Other  Fracture  Support Devices
CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg Female   68         Frontal    AP         1.0                         NaN           NaN           NaN          NaN    NaN            NaN        NaN          NaN           0.0               NaN            NaN       NaN              1.0


In [3]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/input/datasets")

# 1. Inspect NIH directory structure (shallow only)
nih_root = ROOT / "nih-chest-xrays"

print("=== NIH DIRECTORY STRUCTURE ===")

for path in [
    nih_root,
    nih_root / "data"
]:
    print(f"\n{path}")

    if path.exists():
        for item in list(path.iterdir())[:40]:
            print(
                "DIR " if item.is_dir() else "FILE",
                item.name
            )

# 2. Find NIH CSV metadata without scanning images
print("\n=== NIH CSV FILES ===")

nih_csvs = list(nih_root.rglob("*.csv"))

for path in nih_csvs[:30]:
    print(path)

# 3. CheXpert complete metadata statistics
train_csv = ROOT / "ashery/chexpert/train.csv"
valid_csv = ROOT / "ashery/chexpert/valid.csv"

train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("\n=== CHEXPERT DATASET STATISTICS ===")

print("Training rows:", len(train_df))
print("Validation rows:", len(valid_df))

print("\nView distribution:")
print(train_df["Frontal/Lateral"].value_counts(dropna=False))

print("\nTraining label distribution:")
label_columns = [
    c for c in train_df.columns
    if c not in [
        "Path", "Sex", "Age",
        "Frontal/Lateral", "AP/PA"
    ]
]

for col in label_columns:
    print(
        f"{col:32s}",
        "Positive:", int((train_df[col] == 1).sum()),
        "Negative:", int((train_df[col] == 0).sum()),
        "Uncertain:", int((train_df[col] == -1).sum()),
        "Missing:", int(train_df[col].isna().sum())
    )

# 4. Check patient overlap between official train and valid
train_patients = set(
    train_df["Path"].str.extract(r"(patient\d+)")[0].dropna()
)

valid_patients = set(
    valid_df["Path"].str.extract(r"(patient\d+)")[0].dropna()
)

print("\n=== PATIENT SPLIT AUDIT ===")
print("Train unique patients:", len(train_patients))
print("Validation unique patients:", len(valid_patients))
print("Patient overlap:", len(train_patients & valid_patients))

print("\nMetadata audit completed.")


=== NIH DIRECTORY STRUCTURE ===

/kaggle/input/datasets/nih-chest-xrays
DIR  data

/kaggle/input/datasets/nih-chest-xrays/data
FILE ARXIV_V5_CHESTXRAY.pdf
FILE BBox_List_2017.csv
FILE Data_Entry_2017.csv
FILE FAQ_CHESTXRAY.pdf
FILE LOG_CHESTXRAY.pdf
FILE README_CHESTXRAY.pdf
DIR  images_001
DIR  images_002
DIR  images_003
DIR  images_004
DIR  images_005
DIR  images_006
DIR  images_007
DIR  images_008
DIR  images_009
DIR  images_010
DIR  images_011
DIR  images_012
FILE test_list.txt
FILE train_val_list.txt

=== NIH CSV FILES ===
/kaggle/input/datasets/nih-chest-xrays/data/BBox_List_2017.csv
/kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv

=== CHEXPERT DATASET STATISTICS ===
Training rows: 223414
Validation rows: 234

View distribution:
Frontal/Lateral
Frontal    191027
Lateral     32387
Name: count, dtype: int64

Training label distribution:
No Finding                       Positive: 22381 Negative: 0 Uncertain: 0 Missing: 201033
Enlarged Cardiomediastinum       Positive

In [4]:

from pathlib import Path
import pandas as pd
import numpy as np

NIH_ROOT = Path("/kaggle/input/datasets/nih-chest-xrays/data")

# Load NIH metadata
nih_csv = NIH_ROOT / "Data_Entry_2017.csv"
nih_df = pd.read_csv(nih_csv)

print("=" * 65)
print("NIH CHESTXRAY14 METADATA AUDIT")
print("=" * 65)

print("Total records:", len(nih_df))
print("Columns:", nih_df.columns.tolist())
print("Unique patients:", nih_df["Patient ID"].nunique())

print("\nDisease label distribution:")

nih_label_counts = {}

for label in sorted(
    set(
        finding
        for value in nih_df["Finding Labels"].dropna()
        for finding in value.split("|")
    )
):
    count = nih_df["Finding Labels"].apply(
        lambda x: label in str(x).split("|")
    ).sum()

    nih_label_counts[label] = int(count)
    print(f"{label:25s}: {count:,}")

# Official split lists
train_val_file = NIH_ROOT / "train_val_list.txt"
test_file = NIH_ROOT / "test_list.txt"

train_val_images = set(
    train_val_file.read_text().splitlines()
)

test_images = set(
    test_file.read_text().splitlines()
)

print("\n=== OFFICIAL SPLIT AUDIT ===")
print("Train/validation images:", len(train_val_images))
print("Test images:", len(test_images))
print("Image overlap:", len(train_val_images & test_images))

# Map official image lists to metadata
train_val_df = nih_df[
    nih_df["Image Index"].isin(train_val_images)
]

test_df = nih_df[
    nih_df["Image Index"].isin(test_images)
]

train_val_patients = set(train_val_df["Patient ID"])
test_patients = set(test_df["Patient ID"])

print("Train/validation patients:", len(train_val_patients))
print("Test patients:", len(test_patients))
print("Patient overlap:", len(train_val_patients & test_patients))

print("\n=== CHEXPERT FRONTAL AUDIT ===")

chexpert_frontal = train_df[
    train_df["Frontal/Lateral"] == "Frontal"
].copy()

print("Frontal records:", len(chexpert_frontal))
print(
    "Unique frontal patients:",
    chexpert_frontal["Path"].str.extract(
        r"(patient\d+)"
    )[0].nunique()
)

print("\n=== COMMON CANDIDATE LABELS ===")

common_labels = [
    "Cardiomegaly",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion"
]

for label in common_labels:
    print(
        f"{label:25s}",
        "NIH:", nih_label_counts.get(label, 0),
        "| CheXpert positive:",
        int((chexpert_frontal[label] == 1).sum())
    )

print("\nNIH label and split audit completed.")


NIH CHESTXRAY14 METADATA AUDIT
Total records: 112120
Columns: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']
Unique patients: 30805

Disease label distribution:
Atelectasis              : 11,559
Cardiomegaly             : 2,776
Consolidation            : 4,667
Edema                    : 2,303
Effusion                 : 13,317
Emphysema                : 2,516
Fibrosis                 : 1,686
Hernia                   : 227
Infiltration             : 19,894
Mass                     : 5,782
No Finding               : 60,361
Nodule                   : 6,331
Pleural_Thickening       : 3,385
Pneumonia                : 1,431
Pneumothorax             : 5,302

=== OFFICIAL SPLIT AUDIT ===
Train/validation images: 86524
Test images: 25596
Image overlap: 0
Train/validation patients: 28008
Test patients: 2797
Patient overlap: 0

=== CHEXPERT FRONTAL

In [5]:

from pathlib import Path
import pandas as pd
from PIL import Image
import random

NIH_ROOT = Path("/kaggle/input/datasets/nih-chest-xrays/data")
CHEXPERT_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")

nih_df = pd.read_csv(NIH_ROOT / "Data_Entry_2017.csv")
chexpert_df = pd.read_csv(CHEXPERT_ROOT / "train.csv")

# Build NIH image folder list
nih_image_dirs = sorted(
    NIH_ROOT.glob("images_*")
)

def find_nih_image(filename):
    for folder in nih_image_dirs:
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    return None

print("=== NIH IMAGE PATH AUDIT ===")

nih_samples = nih_df.sample(
    min(20, len(nih_df)),
    random_state=42
)

nih_found = 0

for filename in nih_samples["Image Index"]:
    path = find_nih_image(filename)

    if path is not None:
        nih_found += 1
        try:
            with Image.open(path) as img:
                img.verify()
            status = "DECODE OK"
        except Exception as e:
            status = f"DECODE ERROR: {e}"
    else:
        status = "NOT FOUND"

    print(filename, "|", path, "|", status)

print(f"\nNIH sample files found: {nih_found}/20")

# CheXpert path audit
print("\n=== CHEXPERT IMAGE PATH AUDIT ===")

chexpert_frontal = chexpert_df[
    chexpert_df["Frontal/Lateral"] == "Frontal"
].sample(20, random_state=42)

chexpert_found = 0

for relative_path in chexpert_frontal["Path"]:
    parts = Path(relative_path).parts

    # Resolve relative paths against the attached dataset
    if parts[0] == "CheXpert-v1.0-small":
        resolved = CHEXPERT_ROOT / Path(*parts[1:])
    else:
        resolved = CHEXPERT_ROOT / relative_path

    if resolved.is_file():
        chexpert_found += 1

        try:
            with Image.open(resolved) as img:
                img.verify()
            status = "DECODE OK"
        except Exception as e:
            status = f"DECODE ERROR: {e}"
    else:
        status = "NOT FOUND"

    print(relative_path, "|", status)

print(f"\nCheXpert sample files found: {chexpert_found}/20")

print("\nImage path audit completed.")


=== NIH IMAGE PATH AUDIT ===
00022260_003.png | None | NOT FOUND
00012048_007.png | None | NOT FOUND
00014352_005.png | None | NOT FOUND
00006040_001.png | None | NOT FOUND
00011202_002.png | None | NOT FOUND
00002275_007.png | None | NOT FOUND
00003446_008.png | None | NOT FOUND
00030350_000.png | None | NOT FOUND
00004640_000.png | None | NOT FOUND
00008858_007.png | None | NOT FOUND
00018044_033.png | None | NOT FOUND
00004859_000.png | None | NOT FOUND
00014663_049.png | None | NOT FOUND
00027067_002.png | None | NOT FOUND
00026747_000.png | None | NOT FOUND
00017747_058.png | None | NOT FOUND
00006832_004.png | None | NOT FOUND
00025252_060.png | None | NOT FOUND
00021584_003.png | None | NOT FOUND
00010794_003.png | None | NOT FOUND

NIH sample files found: 0/20

=== CHEXPERT IMAGE PATH AUDIT ===
CheXpert-v1.0-small/train/patient18339/study18/view1_frontal.jpg | DECODE OK
CheXpert-v1.0-small/train/patient45651/study4/view1_frontal.jpg | DECODE OK
CheXpert-v1.0-small/train/patient

In [6]:

from pathlib import Path

NIH_ROOT = Path("/kaggle/input/datasets/nih-chest-xrays/data")

for folder in sorted(NIH_ROOT.glob("images_*")):
    print(f"\n📁 {folder.name}")

    try:
        entries = list(folder.iterdir())[:10]

        for item in entries:
            print(
                "  DIR " if item.is_dir() else "  FILE",
                item.name
            )

    except PermissionError:
        print("  Permission denied")

print("\n=== SAMPLE IMAGE SEARCH IN FIRST FOLDER ===")

sample_name = "00022260_003.png"
first_folder = NIH_ROOT / "images_001"

matches = list(first_folder.rglob(sample_name))

for match in matches:
    print("Found:", match)

if not matches:
    print("Sample image not found in images_001.")



📁 images_001
  DIR  images

📁 images_002
  DIR  images

📁 images_003
  DIR  images

📁 images_004
  DIR  images

📁 images_005
  DIR  images

📁 images_006
  DIR  images

📁 images_007
  DIR  images

📁 images_008
  DIR  images

📁 images_009
  DIR  images

📁 images_010
  DIR  images

📁 images_011
  DIR  images

📁 images_012
  DIR  images

=== SAMPLE IMAGE SEARCH IN FIRST FOLDER ===
Sample image not found in images_001.


In [7]:

from pathlib import Path
from PIL import Image
import pandas as pd

NIH_ROOT = Path("/kaggle/input/datasets/nih-chest-xrays/data")

nih_df = pd.read_csv(NIH_ROOT / "Data_Entry_2017.csv")

nih_image_dirs = sorted(
    folder / "images"
    for folder in NIH_ROOT.glob("images_*")
    if (folder / "images").is_dir()
)

def find_nih_image(filename):
    for folder in nih_image_dirs:
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    return None

samples = nih_df.sample(20, random_state=42)

found = 0
decoded = 0

for filename in samples["Image Index"]:
    path = find_nih_image(filename)

    if path is not None:
        found += 1

        try:
            with Image.open(path) as img:
                img.verify()
            decoded += 1
            print("OK:", filename)
        except Exception as e:
            print("DECODE ERROR:", filename, e)
    else:
        print("NOT FOUND:", filename)

print("\n=== NIH PATH AUDIT ===")
print("Image directories:", len(nih_image_dirs))
print("Found:", found, "/ 20")
print("Decoded:", decoded, "/ 20")


OK: 00022260_003.png
OK: 00012048_007.png
OK: 00014352_005.png
OK: 00006040_001.png
OK: 00011202_002.png
OK: 00002275_007.png
OK: 00003446_008.png
OK: 00030350_000.png
OK: 00004640_000.png
OK: 00008858_007.png
OK: 00018044_033.png
OK: 00004859_000.png
OK: 00014663_049.png
OK: 00027067_002.png
OK: 00026747_000.png
OK: 00017747_058.png
OK: 00006832_004.png
OK: 00025252_060.png
OK: 00021584_003.png
OK: 00010794_003.png

=== NIH PATH AUDIT ===
Image directories: 12
Found: 20 / 20
Decoded: 20 / 20


In [8]:

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

NIH_ROOT = Path("/kaggle/input/datasets/nih-chest-xrays/data")
CHEXPERT_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")

OUTPUT_DIR = Path(
    "/kaggle/working/lungvision_disease_classification"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = [
    "Cardiomegaly",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax"
]

# --------------------------------------------------
# 1. NIH IMAGE INDEX
# --------------------------------------------------

print("Building NIH image index...")

nih_image_map = {}

for folder in sorted(NIH_ROOT.glob("images_*")):
    image_dir = folder / "images"

    if image_dir.exists():
        for filename in os.listdir(image_dir):
            if filename.lower().endswith(".png"):
                nih_image_map[filename] = str(
                    image_dir / filename
                )

print("NIH images indexed:", len(nih_image_map))

# --------------------------------------------------
# 2. NIH METADATA AND OFFICIAL SPLIT
# --------------------------------------------------

nih = pd.read_csv(NIH_ROOT / "Data_Entry_2017.csv")

nih_train_val = set(
    (NIH_ROOT / "train_val_list.txt")
    .read_text()
    .splitlines()
)

nih_test = set(
    (NIH_ROOT / "test_list.txt")
    .read_text()
    .splitlines()
)

nih = nih[
    nih["Image Index"].isin(nih_train_val | nih_test)
].copy()

# Keep frontal AP/PA images only
nih = nih[
    nih["View Position"].isin(["AP", "PA"])
].copy()

# Create binary target columns.
# NIH labels are multi-label strings separated by |
for label in LABELS:
    nih[label] = nih["Finding Labels"].apply(
        lambda value: int(
            label in str(value).split("|")
        )
    )

nih["image_path"] = nih["Image Index"].map(
    nih_image_map
)

nih = nih.dropna(subset=["image_path"]).copy()

nih["patient_id"] = nih["Patient ID"].astype(str)

nih["split"] = np.where(
    nih["Image Index"].isin(nih_test),
    "test",
    "train_pool"
)

# Patient-wise internal validation split
nih_pool = nih[nih["split"] == "train_pool"].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(
        nih_pool,
        groups=nih_pool["patient_id"]
    )
)

nih.loc[nih_pool.index[val_idx], "split"] = "validation"
nih.loc[nih_pool.index[train_idx], "split"] = "train"

# --------------------------------------------------
# 3. CHEXPERT METADATA
# --------------------------------------------------

chex_train = pd.read_csv(
    CHEXPERT_ROOT / "train.csv"
)

chex_valid = pd.read_csv(
    CHEXPERT_ROOT / "valid.csv"
)

chex = pd.concat(
    [chex_train, chex_valid],
    ignore_index=True
)

# Frontal images only
chex = chex[
    chex["Frontal/Lateral"] == "Frontal"
].copy()

# Resolve CheXpert paths
def resolve_chexpert_path(relative_path):
    parts = Path(relative_path).parts

    if parts[0] == "CheXpert-v1.0-small":
        return str(
            CHEXPERT_ROOT / Path(*parts[1:])
        )

    return str(CHEXPERT_ROOT / relative_path)

chex["image_path"] = chex["Path"].apply(
    resolve_chexpert_path
)

chex["patient_id"] = chex["Path"].str.extract(
    r"(patient\d+)"
)[0]

chex["official_split"] = np.where(
    chex["Path"].str.contains(
        "/valid/",
        regex=False
    ),
    "official_validation",
    "train_pool"
)

# Preserve original three-state labels.
# Do not convert -1 or NaN to negative.
chex["split"] = chex["official_split"]

chex_pool = chex[
    chex["official_split"] == "train_pool"
].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(
        chex_pool,
        groups=chex_pool["patient_id"]
    )
)

chex.loc[chex_pool.index[train_idx], "split"] = "train"
chex.loc[chex_pool.index[val_idx], "split"] = "validation"

# --------------------------------------------------
# 4. SAVE MANIFESTS
# --------------------------------------------------

nih_path = OUTPUT_DIR / "nih_manifest.csv"
chex_path = OUTPUT_DIR / "chexpert_manifest.csv"

nih.to_csv(nih_path, index=False)
chex.to_csv(chex_path, index=False)

# --------------------------------------------------
# 5. REPORT
# --------------------------------------------------

print("\n" + "=" * 65)
print("NIH MANIFEST")
print("=" * 65)

print("Total images:", len(nih))
print("Unique patients:", nih["patient_id"].nunique())
print("\nSplit distribution:")
print(nih["split"].value_counts())

print("\nNIH disease positives:")
print(nih[LABELS].sum())

print("\n" + "=" * 65)
print("CHEXPERT MANIFEST")
print("=" * 65)

print("Total frontal images:", len(chex))
print("Unique patients:", chex["patient_id"].nunique())
print("\nSplit distribution:")
print(chex["split"].value_counts())

print("\nCheXpert positive labels:")
print((chex[LABELS] == 1).sum())

print("\nCheXpert uncertain labels:")
print((chex[LABELS] == -1).sum())

print("\nCheXpert missing labels:")
print(chex[LABELS].isna().sum())

print("\nSaved manifests:")
print(nih_path)
print(chex_path)

# Verify patient separation in generated splits
for name, frame in [
    ("NIH", nih),
    ("CheXpert", chex[chex["split"].isin(["train", "validation"])])
]:
    train_patients = set(
        frame.loc[frame["split"] == "train", "patient_id"]
    )
    val_patients = set(
        frame.loc[frame["split"] == "validation", "patient_id"]
    )

    print(
        f"\n{name} patient overlap:",
        len(train_patients & val_patients)
    )

print("\nManifest generation completed.")


Building NIH image index...
NIH images indexed: 112120

NIH MANIFEST
Total images: 112120
Unique patients: 30805

Split distribution:
split
train         73023
test          25596
validation    13501
Name: count, dtype: int64

NIH disease positives:
Cardiomegaly      2776
Edema             2303
Consolidation     4667
Pneumonia         1431
Atelectasis      11559
Pneumothorax      5302
dtype: int64

CHEXPERT MANIFEST
Total frontal images: 191229
Unique patients: 64734

Split distribution:
split
train                  161996
validation              29031
official_validation       202
Name: count, dtype: int64

CheXpert positive labels:
Cardiomegaly     23451
Edema            49717
Consolidation    13015
Pneumonia         4683
Atelectasis      29795
Pneumothorax     17700
dtype: int64

CheXpert uncertain labels:
Cardiomegaly      6707
Edema            11818
Consolidation    24381
Pneumonia        15981
Atelectasis      29863
Pneumothorax      2708
dtype: int64

CheXpert missing labels:
Ca

In [9]:

import os
import pandas as pd
import numpy as np

LABELS = [
    "Cardiomegaly",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax"
]

nih = pd.read_csv(
    OUTPUT_DIR / "nih_manifest.csv"
)

chex = pd.read_csv(
    OUTPUT_DIR / "chexpert_manifest.csv"
)

print("=" * 65)
print("NIH DATA INTEGRITY")
print("=" * 65)

nih_missing = (~nih["image_path"].map(os.path.isfile)).sum()

print("Missing image paths:", nih_missing)
print("\nPositive labels by split:")

print(
    nih.groupby("split")[LABELS]
    .sum()
    .astype(int)
)

print("\nImages per split:")
print(nih["split"].value_counts())

print("\n" + "=" * 65)
print("CHEXPERT DATA INTEGRITY")
print("=" * 65)

chex_sample = chex.sample(
    min(1000, len(chex)),
    random_state=42
)

chex_missing = (
    ~chex_sample["image_path"].map(os.path.isfile)
).sum()

print("Sampled paths checked:", len(chex_sample))
print("Missing sampled paths:", chex_missing)

print("\nPositive labels by split:")

for split in ["train", "validation", "official_validation"]:
    subset = chex[chex["split"] == split]

    print(f"\n{split}: {len(subset)} records")
    print((subset[LABELS] == 1).sum().astype(int))

print("\n=== PATIENT SEPARATION ===")

for name, df in [
    ("NIH", nih),
    ("CheXpert", chex[chex["split"].isin(["train", "validation"])])
]:
    train_ids = set(
        df.loc[df["split"] == "train", "patient_id"]
    )

    val_ids = set(
        df.loc[df["split"] == "validation", "patient_id"]
    )

    print(name, "overlap:", len(train_ids & val_ids))

print("\nIntegrity audit completed.")


NIH DATA INTEGRITY
Missing image paths: 0

Positive labels by split:
            Cardiomegaly  Edema  Consolidation  Pneumonia  Atelectasis  \
split                                                                    
test                1069    925           1815        555         3279   
train               1437   1169           2408        737         6935   
validation           270    209            444        139         1345   

            Pneumothorax  
split                     
test                2665  
train               2170  
validation           467  

Images per split:
split
train         73023
test          25596
validation    13501
Name: count, dtype: int64

CHEXPERT DATA INTEGRITY
Sampled paths checked: 1000
Missing sampled paths: 0

Positive labels by split:

train: 161996 records
Cardiomegaly     19865
Edema            42181
Consolidation    10962
Pneumonia         3966
Atelectasis      25189
Pneumothorax     14814
dtype: int64

validation: 29031 records
Cardiome

In [10]:

import os
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Configuration
BATCH_SIZE = 32
IMAGE_SIZE = 224
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Load NIH manifests
nih = pd.read_csv(
    OUTPUT_DIR / "nih_manifest.csv"
)

# Image preprocessing
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class NIHChestXrayDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        image_path = row["image_path"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[LABELS].to_numpy(dtype=np.float32),
            dtype=torch.float32
        )

        return image, labels


# Patient-wise splits already created
train_df = nih[nih["split"] == "train"].copy()
val_df = nih[nih["split"] == "validation"].copy()
test_df = nih[nih["split"] == "test"].copy()

train_dataset = NIHChestXrayDataset(
    train_df,
    transform=train_transform
)

val_dataset = NIHChestXrayDataset(
    val_df,
    transform=eval_transform
)

test_dataset = NIHChestXrayDataset(
    test_df,
    transform=eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nTesting first training batch...")

images, targets = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Target batch shape:", targets.shape)
print("Image dtype:", images.dtype)
print("Target dtype:", targets.dtype)
print("Image range:", images.min().item(), images.max().item())

print("\nSample targets:")
print(pd.DataFrame(
    targets[:5].numpy(),
    columns=LABELS
))

print("\nDataLoader smoke test completed.")


Device: cuda
GPU: Tesla T4

Dataset sizes:
Train: 73023
Validation: 13501
Test: 25596

Testing first training batch...
Image batch shape: torch.Size([32, 3, 224, 224])
Target batch shape: torch.Size([32, 6])
Image dtype: torch.float32
Target dtype: torch.float32
Image range: -2.1179039478302 2.640000104904175

Sample targets:
   Cardiomegaly  Edema  Consolidation  Pneumonia  Atelectasis  Pneumothorax
0           0.0    0.0            0.0        0.0          0.0           0.0
1           0.0    0.0            0.0        0.0          0.0           0.0
2           0.0    0.0            0.0        0.0          0.0           0.0
3           0.0    0.0            0.0        0.0          0.0           0.0
4           0.0    0.0            0.0        0.0          0.0           0.0

DataLoader smoke test completed.


In [11]:

import torch
import torch.nn as nn
from torchvision import models

# Load ImageNet-pretrained DenseNet121
weights = models.DenseNet121_Weights.DEFAULT

model = models.densenet121(weights=weights)

# Replace original 1000-class classifier
in_features = model.classifier.in_features

model.classifier = nn.Linear(
    in_features,
    len(LABELS)
)

model = model.to(DEVICE)

# Multi-label classification loss
criterion = nn.BCEWithLogitsLoss()

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Mixed precision support
use_amp = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

print("=" * 55)
print("DENSENET121 MODEL INITIALIZED")
print("=" * 55)

print("Architecture:", model.__class__.__name__)
print("Input channels:", 3)
print("Output classes:", model.classifier.out_features)
print("Output labels:", LABELS)
print("Loss:", criterion.__class__.__name__)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Mixed precision:", use_amp)

# Verify output dimensions
model.eval()

with torch.no_grad():
    sample = images[:2].to(DEVICE)
    output = model(sample)

print("\nTest input shape:", sample.shape)
print("Model output shape:", output.shape)
print("Model parameter count:", sum(
    p.numel() for p in model.parameters()
))

assert output.shape == (2, 6)

print("\nModel architecture verification passed.")


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 157MB/s]


DENSENET121 MODEL INITIALIZED
Architecture: DenseNet
Input channels: 3
Output classes: 6
Output labels: ['Cardiomegaly', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax']
Loss: BCEWithLogitsLoss
Optimizer: AdamW
Learning rate: 0.0001
Mixed precision: True

Test input shape: torch.Size([2, 3, 224, 224])
Model output shape: torch.Size([2, 6])
Model parameter count: 6960006

Model architecture verification passed.


In [12]:

import numpy as np
import pandas as pd
import torch

# Calculate class distribution using training data only
positive_counts = train_df[LABELS].sum().astype(int)
negative_counts = (
    len(train_df) - positive_counts
).astype(int)

class_distribution = pd.DataFrame({
    "Disease": LABELS,
    "Positive": positive_counts.values,
    "Negative": negative_counts.values,
    "Positive_Rate_%": (
        positive_counts.values / len(train_df) * 100
    ),
    "Positive_Weight": (
        negative_counts.values / positive_counts.values
    )
})

print("=" * 65)
print("NIH TRAINING CLASS DISTRIBUTION")
print("=" * 65)

display(class_distribution.round(3))

# Calculate positive weights
pos_weights = (
    negative_counts.values / positive_counts.values
)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=DEVICE
)

# Update loss with class imbalance weights
criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

print("\nPositive weights:")
for label, weight in zip(LABELS, pos_weights.cpu().numpy()):
    print(f"{label:18s}: {weight:.3f}")

print("\nWeighted BCE loss configured.")


NIH TRAINING CLASS DISTRIBUTION


,Disease,Positive,Negative,Positive_Rate_%,Positive_Weight
0,Cardiomegaly,1437,71586,1.968,49.816
1,Edema,1169,71854,1.601,61.466
2,Consolidation,2408,70615,3.298,29.325
3,Pneumonia,737,72286,1.009,98.081
4,Atelectasis,6935,66088,9.497,9.530
5,Pneumothorax,2170,70853,2.972,32.651



Positive weights:
Cardiomegaly      : 49.816
Edema             : 61.466
Consolidation     : 29.325
Pneumonia         : 98.081
Atelectasis       : 9.530
Pneumothorax      : 32.651

Weighted BCE loss configured.


In [13]:

import os
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import roc_auc_score, average_precision_score

# Training configuration
EPOCHS = 5
PATIENCE = 2
GRAD_CLIP = 1.0

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "densenet121_nih_best.pth"

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

best_val_auc = -np.inf
epochs_without_improvement = 0
history = []

def calculate_metrics(targets, probabilities):

    auc_scores = []
    ap_scores = []

    for i, label in enumerate(LABELS):

        y_true = targets[:, i]
        y_prob = probabilities[:, i]

        if len(np.unique(y_true)) < 2:
            auc = np.nan
            ap = np.nan
        else:
            auc = roc_auc_score(y_true, y_prob)
            ap = average_precision_score(y_true, y_prob)

        auc_scores.append(auc)
        ap_scores.append(ap)

    return auc_scores, ap_scores


for epoch in range(EPOCHS):

    epoch_start = time.time()

    # ---------------- TRAINING ----------------
    model.train()

    running_loss = 0.0
    train_samples = 0

    for batch_images, batch_targets in train_loader:

        batch_images = batch_images.to(
            DEVICE,
            non_blocking=True
        )

        batch_targets = batch_targets.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=use_amp
        ):
            logits = model(batch_images)
            loss = criterion(logits, batch_targets)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * batch_images.size(0)
        train_samples += batch_images.size(0)

    train_loss = running_loss / train_samples

    # ---------------- VALIDATION ----------------
    model.eval()

    val_loss_total = 0.0
    val_samples = 0

    all_targets = []
    all_probabilities = []

    with torch.no_grad():

        for batch_images, batch_targets in val_loader:

            batch_images = batch_images.to(
                DEVICE,
                non_blocking=True
            )

            batch_targets = batch_targets.to(
                DEVICE,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                enabled=use_amp
            ):
                logits = model(batch_images)
                loss = criterion(logits, batch_targets)

            probabilities = torch.sigmoid(logits)

            val_loss_total += loss.item() * batch_images.size(0)
            val_samples += batch_images.size(0)

            all_targets.append(batch_targets.cpu().numpy())
            all_probabilities.append(probabilities.float().cpu().numpy())

    val_loss = val_loss_total / val_samples

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probabilities)

    auc_scores, ap_scores = calculate_metrics(
        y_true,
        y_prob
    )

    macro_auc = float(np.nanmean(auc_scores))
    macro_ap = float(np.nanmean(ap_scores))

    scheduler.step(macro_auc)

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_time = time.time() - epoch_start

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_auc": macro_auc,
        "macro_auprc": macro_ap,
        "learning_rate": current_lr,
        "epoch_time_sec": epoch_time
    })

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("-" * 55)
    print(f"Train loss: {train_loss:.4f}")
    print(f"Validation loss: {val_loss:.4f}")
    print(f"Macro AUROC: {macro_auc:.4f}")
    print(f"Macro AUPRC: {macro_ap:.4f}")
    print(f"Learning rate: {current_lr:.7f}")
    print(f"Time: {epoch_time / 60:.2f} minutes")

    print("\nPer-disease validation metrics:")

    for label, auc, ap in zip(
        LABELS,
        auc_scores,
        ap_scores
    ):
        print(
            f"{label:18s} AUROC={auc:.4f} | AUPRC={ap:.4f}"
        )

    # Save best checkpoint
    if macro_auc > best_val_auc:

        best_val_auc = macro_auc
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_auc": best_val_auc,
            "labels": LABELS,
            "image_size": IMAGE_SIZE
        }, BEST_MODEL_PATH)

        print("\nBest checkpoint saved:", BEST_MODEL_PATH)

    else:
        epochs_without_improvement += 1
        print("\nNo validation AUROC improvement.")

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

# Save training history
history_df = pd.DataFrame(history)

history_df.to_csv(
    OUTPUT_DIR / "nih_training_history.csv",
    index=False
)

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print("Best validation macro AUROC:", round(best_val_auc, 4))
print("Best checkpoint:", BEST_MODEL_PATH)
display(history_df)



Epoch 1/5
-------------------------------------------------------
Train loss: 1.1057
Validation loss: 1.1322
Macro AUROC: 0.8088
Macro AUPRC: 0.1378
Learning rate: 0.0001000
Time: 47.14 minutes

Per-disease validation metrics:
Cardiomegaly       AUROC=0.8791 | AUPRC=0.2041
Edema              AUROC=0.8608 | AUPRC=0.0877
Consolidation      AUROC=0.7757 | AUPRC=0.0985
Pneumonia          AUROC=0.7625 | AUPRC=0.0322
Atelectasis        AUROC=0.7787 | AUPRC=0.2779
Pneumothorax       AUROC=0.7958 | AUPRC=0.1266

Best checkpoint saved: /kaggle/working/lungvision_disease_classification/checkpoints/densenet121_nih_best.pth

Epoch 2/5
-------------------------------------------------------
Train loss: 1.0063
Validation loss: 1.1030
Macro AUROC: 0.8143
Macro AUPRC: 0.1620
Learning rate: 0.0001000
Time: 37.37 minutes

Per-disease validation metrics:
Cardiomegaly       AUROC=0.8950 | AUPRC=0.2310
Edema              AUROC=0.8751 | AUPRC=0.1342
Consolidation      AUROC=0.7929 | AUPRC=0.1128
Pneumonia 

,epoch,train_loss,val_loss,macro_auc,macro_auprc,learning_rate,epoch_time_sec
0,1,1.105683,1.132232,0.808754,0.137820,0.0001,2828.587773
1,2,1.006306,1.102983,0.814312,0.162032,0.0001,2242.214946
2,3,0.933060,1.266051,0.801935,0.159012,0.0001,2068.863248
3,4,0.847063,1.333251,0.820582,0.166984,0.0001,2029.730595
4,5,0.710669,1.694733,0.817196,0.164999,0.0001,1983.076374
